# 05 · Methods behind Brief C · Group 10

We first checked H1 scheduled volume, cleaned and profiled the data, and used Brief A/B as exploratory work. Group 10 selected Brief C as the board-facing finance question.

Active scheduled test lines grew 4.97% (rounds to 5%): 457,539 in H1 2026 versus 435,880 in H1 2025, against 18.32% implied plan growth. A/B methods remain as supporting context; the Brief C bridge is the final analysis.

**Needs:** notebook 03 for booking exploration. The finance bridge cleans its own invoice, gateway and settlement tables from the data pack.

## CONFIG

In [ ]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "Brief C",
    "group": "10",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-07-01"),
    "period_last": ("2025-01-01", "2025-07-01"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

bk = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["slot_at", "collected_at"], dtype={"booking_id": "string", "person_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
is_last = bk["slot_at"].ge(pd.Timestamp(CONFIG["period_last"][0])) & bk["slot_at"].lt(pd.Timestamp(CONFIG["period_last"][1]))
is_this = bk["slot_at"].ge(pd.Timestamp(CONFIG["period_this"][0])) & bk["slot_at"].lt(pd.Timestamp(CONFIG["period_this"][1]))
bk["half"] = np.select([is_last, is_this], ["H1_2025", "H1_2026"], default="other")
bk = bk[bk["half"] != "other"]
print(bk.groupby("half").agg(bookings=("booking_id", "size"), tests=("n_tests", "sum"), people=("person_id", "nunique")))

## 1 · The tree (brief A)

Tests = people × visits per person × tests per visit. Growth in a product is close to the sum of growths in the factors (exactly so in logs). Compute each factor for both halves and check that they multiply back.

In [ ]:
def tree(df, by=None):
    g = df.groupby(([by] if by else []) + ["half"], observed=True).agg(tests=("n_tests", "sum"), visits=("booking_id", "size"), people=("person_id", "nunique"))
    g["visits_per_person"] = g["visits"] / g["people"]
    g["tests_per_visit"] = g["tests"] / g["visits"]
    return g

t = tree(bk)
growth = (t.xs("H1_2026") / t.xs("H1_2025") - 1) * 100
print(growth.round(2))
# log-decomposition: the branch growths sum to the total growth exactly
logs_ = np.log(t.xs("H1_2026") / t.xs("H1_2025"))
checks.check("branches multiply back to the total (log decomposition)", abs(logs_["people"] + logs_["visits_per_person"] + logs_["tests_per_visit"] - logs_["tests"]) < 1e-9,
             f"people {logs_['people']:+.4f} + visits/person {logs_['visits_per_person']:+.4f} + tests/visit {logs_['tests_per_visit']:+.4f} = {logs_['tests']:+.4f}")

In [ ]:
by_city = tree(bk, "city")
city_growth = ((by_city.xs("H1_2026", level="half") / by_city.xs("H1_2025", level="half") - 1) * 100).round(1)
city_growth[["tests", "people", "visits_per_person", "tests_per_visit"]].sort_values("tests")

> **Interview angle.** Revenue grew 10 percent and patients grew 4. Decompose the rest without a calculator, then say what you would check before presenting it.
>
> *What a good answer covers:* Roughly 6 points sit in visits per patient and revenue per visit together; log growth makes the split exact. Check that the patient count is a count of people and not of ids (duplicates), that both periods use the same definition of a visit, and that a system change did not move the grain.

## 2 · The ladder (brief B)

Confirm the drop, compare like with like, decompose, isolate, hypothesise. In code the ladder is a sequence of groupbys where each step changes one thing: the grain, the window, the segment.

In [ ]:
bk["quarter"] = bk["slot_at"].dt.to_period("Q").astype(str)
q = bk[bk["city"].isin(["Chennai", "Jaipur"])].groupby(["city", "quarter", "system"], observed=True).agg(bookings=("booking_id", "size"), tests=("n_tests", "sum")).reset_index()
pivot = q.pivot_table(index=["city", "quarter"], columns="system", values=["bookings", "tests"], aggfunc="sum", fill_value=0)
pivot

In [ ]:
# Rung 2, like with like: the same quarter a year earlier, per city, at booking grain, after the identity rule
yoy = bk[bk["city"].isin(["Chennai", "Jaipur"])].groupby(["city", "quarter"], observed=True)["booking_id"].size().unstack("quarter")
def pct_change(table, later, earlier, label):
    """Like with like means both periods must exist; say so when one does not, rather than fail quietly."""
    if later in table and earlier in table:
        print(((table[later] / table[earlier] - 1) * 100).round(1).rename(label))
    else:
        print(f"{label}: period {earlier if earlier not in table else later} is not in this pack (sample data?); nothing to compare yet")
pct_change(yoy, "2026Q2", "2025Q2", "Q2 2026 vs Q2 2025, bookings, %")

## 3 · Brief C invoice-to-capture bridge

Invoice date defines the H1 cohort; payment status and India-time timestamps define captures at close and late receipts through 15 July. Counts and exact rupees are written to the audit outputs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "clean_brief_c.py"), "--source-dir", str(ROOT / "data" / "Build1_DataPack.zip"), "--output-dir", str(CONFIG["clean"] / "brief_c")], cwd=ROOT, check=True)
cmd = [sys.executable, str(ROOT / "analyze_brief_c_reconciliation.py"), "--data-dir", str(CONFIG["clean"] / "brief_c"), "--output-dir", str(ROOT / "outputs" / "brief_c"), "--action-dir", str(CONFIG["clean"] / "brief_c" / "actions"), "--year", "2026", "--observation-cutoff", "2026-07-15"]
subprocess.run(cmd, cwd=ROOT, check=True)
pd.read_csv(ROOT / "outputs" / "brief_c" / "finance_bridge_summary.csv", dtype=str)
for filename in ("finance_outstanding_by_invoice_month.csv", "finance_outstanding_by_city_channel.csv", "finance_open_balance_age.csv", "finance_unmatched_capture_breakdown.csv"):
    print(f"\n{filename}")
    print(pd.read_csv(ROOT / "outputs" / "brief_c" / filename, dtype=str).to_string(index=False))

> **Interview angle.** Finance says collected is 4 percent below invoiced. Before you call it a leak, what three things do you separate it into?
>
> *What a good answer covers:* Timing (invoiced in the period, paid after the cutoff), unmatched (payments whose reference does not resolve to an invoice, and invoices with no payment at all), and amount differences (fees, partial payments, retries). Each is a bar with a count and an amount, and only the middle one is a possible leak.

## 5 · The fair comparison (brief E)

Reproduce marketing's number, then look at who got the offer, then split. The grid: offer or not, before or after, by segment. A lift that holds in every segment is a lift; one that holds only in the total is a mix shift.

In [ ]:
camp = pd.read_csv(CONFIG["raw"] / "campaigns" / "campaign_bookings.csv.gz", dtype="string")
camp["booking_id"] = camp["booking_id"].str.strip()
bkq = bk.merge(camp[["booking_id", "campaign_id"]], on="booking_id", how="left", validate="one_to_one")
bkq["offer"] = bkq["campaign_id"].notna()
all_b = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["slot_at"], dtype={"booking_id": "string", "person_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
all_b["quarter"] = all_b["slot_at"].dt.to_period("Q").astype(str)
qq = all_b.groupby("quarter")["booking_id"].size()
r = lambda a, b: f"{(qq[a]/qq[b]-1)*100:+.1f}%" if a in qq and b in qq else "period missing in this pack"
print("marketing's comparison, Q1 2026 vs Q4 2025:", r("2026Q1", "2025Q4"), " | the same comparison a year earlier is not in the pack; Q1 2025 vs Q2 2025:", r("2025Q1", "2025Q2"))
# who got the offer: first-seen date per person, so 'new' is defined before the offer and cannot be changed by it
first_seen = all_b.groupby("person_id")["slot_at"].min()
bkq["is_new_patient"] = bkq["person_id"].map(first_seen) >= pd.Timestamp("2026-01-01")
pd.crosstab(bkq.loc[bkq["half"] == "H1_2026", "offer"], bkq.loc[bkq["half"] == "H1_2026", "is_new_patient"], normalize="index").round(3)

In [ ]:
seg = all_b.groupby(["quarter", "city"])["booking_id"].size().unstack("quarter")
pct_change(seg, "2026Q1", "2025Q4", "Q1 2026 vs Q4 2025 by city, %")
print("\nIf every city shows about the same lift with and without the offer, the lift is the season, not the offer. Split by channel next.")

> **Interview angle.** Marketing shows bookings up 9 percent in the campaign quarter. Give three reasons that is not proof the campaign worked, and the comparison that would be.
>
> *What a good answer covers:* Seasonality (the same quarter rises every year), selection (the offer reached people who were already going to book), and mix (new patients grew for other reasons). The comparison that would show cause is like-for-like patients who did not get the offer in the same period: a holdout, or at least a matched before-behaviour comparison.

## 6 · SUM: the chart for slide 4

One chart, the claim's number against its comparison, with the denominator in the axis label. Save it to `slides/`.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4.2))
g = city_growth["tests"].sort_values()
ax.barh(g.index, g.values, color=["#D63A6A" if v < 0 else "#4E38AB" for v in g.values])
ax.axvline(0, color="#1A1440", lw=1)
ax.set_xlabel("Active scheduled test lines, H1 2026 vs H1 2025 (denominator: H1 2025 scheduled lines)")
ax.set_title("Where the growth went, by city")
for i, v in enumerate(g.values): ax.text(v + (0.3 if v >= 0 else -0.3), i, f"{v:+.1f}%", va="center", ha="left" if v >= 0 else "right", fontsize=9)
plt.tight_layout()
(ROOT / "slides").mkdir(exist_ok=True)
fig.savefig(ROOT / "slides" / "slide4_chart.png", dpi=200)
print("saved slides/slide4_chart.png")
checks.summary()

## Brief C summary · short answers

- **What remained open?** ₹1.947 crore across 11,586 invoices at 30 June; ₹1.648 crore across 9,824 invoices at 15 July. Later captures on 1,763 no-capture invoices corresponded to ₹0.298 crore of invoice face value.
- **Which captures lack a safe match?** 1,541 H1 captures totaling ₹2,668,046 (₹0.267 crore); 959 had the standard reference shape but no safe match, while 582 had a 7-digit final segment.
- **What must Finance confirm?** Source completeness, contractual due dates, refunds/reversals, settlement-to-bank matching, and independent clinic cash coverage. Invoice-age bands are not an overdue report.